In [20]:
import pandas as pd
import numpy as np
import re

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier


import warnings

from neural_network.titanic_dataset import TitanicDataset
from neural_network.titanic_neural_network import TitanicNeuralNetwork

warnings.filterwarnings('ignore')


In [33]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [22]:
def get_train_fare_median(df: pd.DataFrame):
    return df['Fare'].median()

def get_train_age_median(df: pd.DataFrame):
    return df['Age'].median()

def add_initial_feature(df: pd.DataFrame):
    df['Initial'] = 0
    for _ in df:
        df['Initial'] = df['Name'].str.extract(r'([A-Za-z]+)\.')

    df['Initial'] = df['Initial'].replace(
        ['Mlle','Mme','Ms','Dr','Major','Lady','Countess','Jonkheer','Col','Rev','Capt','Sir','Don','Dona'],
        ['Miss','Miss','Miss','Mr','Mr','Mrs','Mrs','Other','Other','Other','Mr','Mr','Mr','Mrs']
    )

    df['Initial'] = df['Initial'].replace({
        'Mr': 0,
        'Mrs': 1,
        'Miss': 2,
        'Master': 3,
        'Other': 4
    }).astype(int)

    return df

def preprocessing(df: pd.DataFrame, fare_median, age_median):
    df['Age'] = df['Age'].fillna(age_median)

    df['Cabin'] = df['Cabin'].fillna("U0")

    deck = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}

    df['Deck'] = df['Cabin'].map(lambda x: re.compile("([a-zA-Z]+)").search(x).group())
    df['Deck'] = df['Deck'].map(deck)

    df['Deck'] = df['Deck'].fillna(0)

    df['Deck'] = df['Deck'].astype(int)

    df['Embarked'] = df['Embarked'].fillna('S')
    df['Fare'] = (df['Fare'].fillna(fare_median))

    df['FamilySize'] = 0
    df['FamilySize'] = df['Parch'] + df['SibSp'] + 1

    df['FarePerPerson'] = df['Fare'] / df['FamilySize']

    df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 60, np.inf], labels=False) + 1

    df['Alone'] = 0
    df.loc[df['FamilySize'] == 1, 'Alone'] = 1

    df['Sex'] = df['Sex'].replace({'male': 0, 'female': 1}).astype(int)
    df['Embarked'] = df['Embarked'].replace({'S': 0, 'C': 1, 'Q': 2}).astype(int)
    df['AgeGroup'] = df['AgeGroup'].replace({'Child': 1, 'Teenager': 2, 'Adult': 3, 'Senior': 4})

    df['WomanHighClass'] = ((df['Sex'] == 1) & (df['Pclass'] <= 2)).astype(int)
    df['ManThirdClass'] = ((df['Sex'] == 0) & (df['Pclass'] == 3)).astype(int)

    df = df.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1)

    return df

X_train = add_initial_feature(train)
X_test = add_initial_feature(test)
fare_median = get_train_fare_median(train)
age_median = get_train_age_median(train)
X_train = preprocessing(train, fare_median, age_median)
y = X_train.pop("Survived")

test_ids = test["PassengerId"]
X_test = preprocessing(test, fare_median, age_median)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [23]:

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, solver="lbfgs"))
])

model = GridSearchCV(
    estimator=pipe,
    param_grid={
        "classifier__C": [0.0001, 0.001, 0.01, 0.1],
        "classifier__penalty": ["l2", "l1"],
        "classifier__solver": ["liblinear", "saga"]
    },
    cv=cv,
    scoring="roc_auc"
)

model.fit(X_train, y)

print(model.best_params_)
print(model.best_score_)

predictions = model.predict(X_test)


{'classifier__C': 0.01, 'classifier__penalty': 'l2', 'classifier__solver': 'saga'}
0.8641889251983927


In [24]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC())
])

model = GridSearchCV(
    estimator=pipe,
    param_grid={
        "classifier__kernel": ["linear", "rbf"],
        "classifier__C": [0.001, 0.01, 0.05, 0.1, 0.5, 1],
        "classifier__gamma": ["scale", "auto"]
    },
    cv=cv,
    scoring="accuracy"
)

model.fit(X_train, y)


print(model.best_params_)
print(model.best_score_)

predictions = model.predict(X_test)

{'classifier__C': 1, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
0.8372355784319879


In [25]:
pipe = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [2, 3, 4, 5, 6, 8, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy"]
}

model = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc"
)


model.fit(X_train, y)

print("Best params:", model.best_params_)
print("Best CV score:", model.best_score_)

predictions = model.predict(X_test)


Best params: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 10, 'min_samples_split': 2}
Best CV score: 0.8749863981564634


In [26]:
rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

model = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)
model.fit(X_train, y)

print("Best params:", model.best_params_)
print("Best CV score:", model.best_score_)

predictions = model.predict(X_test)


Best params: {'bootstrap': True, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Best CV score: 0.8847393359200298


In [28]:
gb = GradientBoostingClassifier(random_state=42)

param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0],
    "min_samples_split": [2, 5, 10]
}

model = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

model.fit(X_train, y)

print("Best params:", model.best_params_)
print("Best CV score:", model.best_score_)

predictions = model.predict(X_test)


Best params: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 100, 'subsample': 0.8}
Best CV score: 0.8473416609126859


In [29]:
xgb = XGBClassifier(random_state=42)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 0.1, 0.3],
    "min_child_weight": [1, 5, 10]
}

model = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv = cv,
    scoring="roc_auc",
    n_jobs=-1
)

model.fit(X_train, y)

print("Best params:", model.best_params_)
print("Best CV score:", model.best_score_)

predictions = model.predict(X_test)


Best params: {'colsample_bytree': 0.8, 'gamma': 0.1, 'learning_rate': 0.1, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 200, 'subsample': 0.8}
Best CV score: 0.8855385259953058


In [30]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

rf_best = RandomForestClassifier(
    bootstrap=True,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=5,
    n_estimators=200,
    random_state=42
)

xgb_best = XGBClassifier(
    colsample_bytree=0.8,
    gamma=0.1,
    learning_rate=0.1,
    max_depth=4,
    min_child_weight=5,
    n_estimators=200,
    subsample=0.8,
    random_state=42
)

lr_best = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(C=0.01, penalty='l2', solver='saga', max_iter=1000))
])

voting = VotingClassifier(
    estimators=[
        ('rf', rf_best),
        ('xgb', xgb_best),
        ('lr', lr_best)
    ],
    voting='soft'
)

voting.fit(X_train, y)

predictions = model.predict(X_test)



In [31]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch


scaler = StandardScaler()

X_train_nn, X_val_nn, y_train_nn, y_val_nn = train_test_split(
    X_train, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_nn = scaler.fit_transform(X_train_nn)
X_val = scaler.transform(X_val_nn)

train_dataset = TitanicDataset(X_train_nn, y_train_nn.values)
val_dataset = TitanicDataset(X_val, y_val_nn.values)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


model = TitanicNeuralNetwork(input_dim=X_train_nn.shape[1]).to(device)


criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(21):

    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    correct = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            pred = model(X_batch)
            val_loss += criterion(pred, y_batch).item()
            correct += ((pred > 0.5) == y_batch).float().sum()


    if epoch % 10 == 0:
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        accuracy = correct / len(val_loader.dataset)
        print(f"Epoch {epoch:3d} | train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f} | val_acc: {accuracy:.4f}")


predictoins = []

X_test_scaled = scaler.transform(X_test)

test_dataset = TitanicDataset(X_test_scaled)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()
predictions = []

with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        pred = model(X_batch)
        predictions.extend((pred > 0.5).int().cpu().numpy())




Epoch   0 | train_loss: 0.6391 | val_loss: 0.6099 | val_acc: 0.7542
Epoch  10 | train_loss: 0.3734 | val_loss: 0.4301 | val_acc: 0.8156
Epoch  20 | train_loss: 0.3786 | val_loss: 0.4325 | val_acc: 0.8212


In [32]:
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)
